# Practice: ALE

In the lesson “ALE” we went through why the PDP goes wrong on correlated features, how ALE fixes that with prediction differences inside intervals, and computed ALE by hand on 19 shoppers. In this notebook ALE is computed first on the numbers of the lesson, then on real data, checked on a linear model, and compared with the PDP and the M-plot on a synthetic example where the right answer is known in advance.

Every task has a cell with `...` that you fill in. The self-check quizzes on this notebook are in the lesson “ALE: Practice” of the module “Explanation Methods for Machine Learning Models pt. 1” on the course platform.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.inspection import partial_dependence

%config InlineBackend.figure_format = 'retina'
BLUE, ORANGE, AQUA, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#8a8983", "#d8d7d2"
plt.rcParams.update({"axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.alpha": 0.6,
                     "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED, "figure.dpi": 100})

## 1. ALE on the numbers of the lesson

In the step “ALE from scratch” of the lesson “ALE” the age of 19 shoppers is split into three intervals of 6, 7 and 6 observations. For interval 2 the model gave spending predictions at the lower border (age 30) and at the upper one (age 37) — they are in the cell below. The local effects of intervals 1 and 3 from the lesson “ALE”: $\Delta_1 = 19.33$, $\Delta_3 = 7.67$.

**Task 1.** Compute $\Delta_2$ (round to hundredths, as in the lesson), the accumulated ALE $A_k = \sum_{l \le k} \Delta_l$ and the centered ALE

$$A_k - \bar A,\qquad \bar A = \frac{1}{n}\sum_{k} |N(k)|\, A_k.$$

In [ ]:
pred_lower = np.array([61, 40, 40, 88, 70, 39, 61])  # age = 30
pred_upper = np.array([34, 76, 44, 76, 51, 98, 58])  # age = 37
counts = np.array([6, 7, 6])

delta_2 = ...  # your code: the mean prediction difference at the borders of interval 2, to hundredths
delta = np.array([19.33, delta_2, 7.67])
A = ...      # your code: the accumulated ALE
A_bar = ...  # your code: the mean of A weighted by the number of objects in the intervals
print("Delta_2 =", delta_2)
print("A =", A.round(2), " A_bar =", round(A_bar, 2))
print("centered ALE =", (A - A_bar).round(2))

## 2. ALE on real data



### Data and model

The California Housing dataset ships with `sklearn`: 20,640 California block groups from the 1990 census, eight block-group features, and the target is the median house value in the block group in hundreds of thousands of dollars. The features: `MedInc` — median income (tens of thousands of dollars), `HouseAge` — median house age, `AveRooms` and `AveBedrms` — average number of rooms and bedrooms per house, `Population` — block-group population, `AveOccup` — average number of occupants per house, `Latitude` and `Longitude`.

We split the data into a training and a test part (80% and 20%, `random_state=42`) and train a gradient boosting model `HistGradientBoostingRegressor` with default parameters. For the methods of this notebook the model is a black box: all they need from it is the prediction `model.predict`.

In [ ]:
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = HistGradientBoostingRegressor(random_state=0).fit(X_train, y_train)
print(f"Test R^2 of the model: {model.score(X_test, y_test):.3f}")
X.describe().T.round(2)

We split the feature's range into $K$ intervals by quantiles, so that every interval gets about the same number of objects. Neighboring intervals share a border $z_k$. For an object of interval $k$ we take the difference of the predictions at the right and the left border, leaving the object's other features as they are; the mean over the objects of the interval is the local effect $\Delta_k$. The curve is accumulated from the left end, $A_0 = 0$, and centered so that its mean over the objects is zero (the value of an object in an interval is the midpoint between $A_{k-1}$ and $A_k$). This is how the packages `PyALE` and `ALEPlot` center; the paper by Apley and Zhu (equation 16) subtracts $\frac1n\sum_k |N(k)|\,A_k$, and the curves differ only by a constant shift.

**Task 2.** Complete the function `ale`. It returns the interval borders `z` and the centered ALE values at those borders. Check with a counter that ALE costs $2n$ predictions for any $K$.

In [ ]:
def ale(model, X, feature, K=20):
    x = X[feature].values
    z = np.unique(np.quantile(x, np.linspace(0, 1, K + 1)))
    k = np.clip(np.searchsorted(z, x, side="left") - 1, 0, len(z) - 2)  # interval index of each object
    lo, hi = X.copy(), X.copy()
    lo[feature], hi[feature] = z[k], z[k + 1]
    n_k = np.bincount(k, minlength=len(z) - 1)
    diff = ...   # your code: the prediction difference at the right and left borders
    delta = ...  # your code: the mean of diff over the objects of each interval
    A = ...      # your code: the accumulated curve of length len(z), A[0] = 0
    mid = (A[:-1] + A[1:]) / 2
    return z, A - (mid * n_k).sum() / n_k.sum()


class CountingModel:
    def __init__(self, model):
        self.model, self.rows = model, 0

    def predict(self, X):
        self.rows += len(X)
        return self.model.predict(X)


for K in [10, 50]:
    counted = CountingModel(model)
    ale(counted, X_test, "MedInc", K=K)
    print(f"K = {K}: predictions {counted.rows}, 2n = {2 * len(X_test)}")

## 3. ALE of a linear model

The lesson “ALE” showed: for a linear model the prediction difference at the interval borders equals $w_j (z_k - z_{k-1})$ for every object, and ALE is a straight line with slope $w_j$ however the features are related.

**Task 3.** Train a `LinearRegression`, compute ALE for all eight features, estimate the slope of each curve (`np.polyfit` of degree 1) and compare with the weights `lin.coef_`.

In [ ]:
lin = LinearRegression().fit(X_train, y_train)
rows = {}
for j, feature in enumerate(X.columns):
    z, a = ale(lin, X_test, feature)
    slope = ...  # your code: the slope of a line fitted to the ALE
    rows[feature] = {"ALE slope": slope, "model weight": lin.coef_[j]}
pd.DataFrame(rows).T

In [ ]:
z, a = ale(lin, X_test, "MedInc")
w = lin.coef_[0]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(z, a, color=BLUE, lw=2, marker="o", ms=4, label="ALE")
ax.plot(z, w * (z - z.mean()) + a.mean(), color=ORANGE, lw=1.2, ls="--", label=f"a line with slope w = {w:.3f}")
for q in np.quantile(X_test["MedInc"], np.linspace(0.1, 0.9, 9)):
    ax.axvline(q, ymin=0, ymax=0.04, color=MUTED, lw=1)
ax.set_xlabel("MedInc — median income of the block group, tens of thousands of dollars")
ax.set_ylabel("ALE, hundreds of thousands of dollars")
ax.set_title("ALE for MedInc of a linear regression on California Housing")
ax.legend()
plt.show()

## 4. ALE of the boosting model

**Task 4.** Draw the ALE of the gradient boosting model for the features `MedInc`, `AveOccup`, `HouseAge`, `Latitude` on the test data, $K = 20$. Mark the feature's deciles at the bottom of each plot and limit the $x$ axis to the 1st and 99th percentiles: for `AveOccup` the last interval stretches to 1243, and without the limit the other intervals merge at the left edge. Compute the span of each curve (maximum minus minimum).

In [ ]:
features = ["MedInc", "AveOccup", "HouseAge", "Latitude"]
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharey=True)
span = {}
for ax, feature in zip(axes.ravel(), features):
    z, a = ...  # your code: the boosting model's ALE for the feature
    span[feature] = a.max() - a.min()
    ax.axhline(0, color=GRID, lw=1)
    ax.plot(z, a, color=BLUE, lw=2, marker="o", ms=3)
    for q in np.quantile(X_test[feature], np.linspace(0.1, 0.9, 9)):
        ax.axvline(q, ymin=0, ymax=0.04, color=MUTED, lw=1)
    ax.set_xlim(X_test[feature].quantile(0.01), X_test[feature].quantile(0.99))
    ax.set_xlabel(feature)
    ax.set_ylabel("ALE, hundreds of thousands of dollars")
fig.suptitle("ALE for California Housing, gradient boosting, K = 20; ticks at the bottom are the feature deciles, the x axis spans the 1st to 99th percentile")
fig.tight_layout()
plt.show()
pd.Series(span, name="ALE span").round(3)

## 5. PDP, M-plot and ALE on correlated features

Take a model that needs no training: $f(x_1, x_2) = x_1 + x_2 + 5(x_1 - x_2)^2$. The features almost coincide: $x_1 \sim U(-1, 1)$, $x_2 = x_1 + \varepsilon$, $\varepsilon \sim N(0, 0.05^2)$. Inside the data $x_1 \approx x_2$, the third term is almost zero, and the own effect of $x_1$ is a straight line with slope 1. Outside the data, where $x_1$ and $x_2$ are far apart, the third term is large.

**Task 5.** Compute three curves for $x_1$ on a grid of 10 points from $-0.9$ to $0.9$: the PDP, the M-plot (the average prediction of the objects with $|x_1 - v| < 0.1$) and ALE with $K = 10$. Fit a parabola to each curve and compare its coefficients with the true ones: 0 for $x_1^2$ and 1 for $x_1$.

In [ ]:
rng = np.random.RandomState(0)
x1 = rng.uniform(-1, 1, 2000)
D = pd.DataFrame({"x1": x1, "x2": x1 + rng.normal(0, 0.05, 2000)})


class KnownModel:
    def predict(self, X):
        return X["x1"].values + X["x2"].values + 5 * (X["x1"].values - X["x2"].values) ** 2


known = KnownModel()
v = np.linspace(-0.9, 0.9, 10)
pdp = ...    # your code: the average prediction over all rows of D with x1 replaced by t
mplot = ...  # your code: the average prediction of the rows of D with |x1 - t| < 0.1
z, a = ale(known, D, "x1", K=10)

for name, xs, curve in [("PDP", v, pdp), ("M-plot", v, mplot), ("ALE", z, a)]:
    c2, c1, _ = np.polyfit(xs, curve, 2)
    print(f"{name}: coefficient of x1^2 {c2:.3f}, of x1 {c1:.3f}")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(v, pdp - pdp.mean(), color=ORANGE, lw=2, label="PDP")
ax.plot(v, mplot - mplot.mean(), color=AQUA, lw=2, label="M-plot")
ax.plot(z, a, color=BLUE, lw=2, label="ALE")
ax.plot(v, v, color=INK, lw=1, ls="--", label="true effect of x1")
ax.set_xlabel("x1")
ax.set_ylabel("effect, centered")
ax.set_title("f = x1 + x2 + 5(x1 - x2)^2, x2 = x1 + noise")
ax.legend()
plt.show()

**Space for your conclusions:** which of the three curves recovers the own effect of $x_1$? What exactly breaks each of the other two?

## 6. Choosing the number of intervals $K$

A local effect is a mean over $|N(k)|$ objects: the larger $K$, the fewer objects per interval and the noisier the curve; the smaller $K$, the coarser the curve.

**Task 6.** Draw the boosting model's ALE for `AveOccup` on 1000 test rows with $K = 5$, $20$ and $200$. For each $K$ compute the average number of objects per interval and the total variation of the curve — the sum of absolute increments $\sum_k |A_k - A_{k-1}|$.

In [ ]:
X_k = X_test.sample(1000, random_state=0)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
rows = {}
for K, color in [(5, ORANGE), (20, BLUE), (200, AQUA)]:
    z, a = ale(model, X_k, "AveOccup", K=K)
    rows[K] = ...  # your code: the average number of objects per interval and the total variation
    ax.plot(z, a, color=color, lw=1.5, label=f"K = {K}")
ax.set_xlim(X_k["AveOccup"].quantile(0.01), X_k["AveOccup"].quantile(0.99))
ax.set_xlabel("AveOccup")
ax.set_ylabel("ALE, hundreds of thousands of dollars")
ax.set_title("ALE for AveOccup with different K, 1000 test rows")
ax.legend()
plt.show()
pd.DataFrame(rows).T.round(3)

**Space for your conclusions:** how does the curve change as $K$ grows, and which $K$ would you choose for these 1000 rows?